![Project](https://img.shields.io/badge/NS%20Health%20%26%20Population%20Analytics-1B3A6B?style=for-the-badge&logoColor=white)

# 🎓 Education Attainment by Age Group & Gender — ETL Pipeline
![Python](https://img.shields.io/badge/Python_3.13-C9A020?style=flat-square&logo=python&logoColor=white) ![pandas](https://img.shields.io/badge/pandas-1A6B5A?style=flat-square&logo=pandas&logoColor=white) ![Source](https://img.shields.io/badge/Source-Statistics%20Canada-1B3A6B?style=flat-square)

> **Analytical Question:** How does educational attainment in Nova Scotia compare to the national average? How have attainment rates changed across age groups and genders from 2012 to 2025?

This notebook extracts and cleans **Statistics Canada Table 37-10-0130-01 — Educational Attainment of the Population Aged 25 to 64, by Age Group and Gender** from 8 raw StatsCan CSV exports (4 age groups × 2 genders) and stacks them into a single time-series dataset.

---

## 📦 Source Data

| Files | Content | Years |
|-------|---------|-------|
| `Men Education levels (1–5).csv` × 4 | Attainment by age group — Men | 2012–2025 |
| `Women Education Levels (1–4).csv` × 4 | Attainment by age group — Women | 2012–2025 |

**Source:** Statistics Canada — Labour Force Survey (LFS), Table 37-10-0130-01  
**Geography:** Canada, all provinces and territories

---

## 🔧 ETL Pipeline Architecture

```
DATA_DIR.glob("*.csv")
    └── for each .csv file:
          read_statscan_table()   ← extract age group, gender + locate data block
          process_file()          ← clean + forward-fill + melt wide→long + tag metadata
    └── pd.concat() → drop_duplicates() → sort → export CSV + SQLite
```

---

## ⚠️ Known Data Challenges

| Challenge | Detail | Solution |
|-----------|--------|----------|
| Metadata header block | Title, frequency, footnotes precede the data table | Scan for row containing `'Geography'` and `'Education'` |
| `,,Percent,,` sub-header | StatsCan inserts a unit row immediately after the column header | Skip rows matching `^,+"Percent"` before parsing |
| Geography forward-fill | Province name only appears on first of 3 education-level rows | `ffill()` after replacing empty strings with `NaN` |
| Footnote numbers on geography | `'Canada 11'`, `'Below upper secondary 12'` | Regex strip trailing digits |
| Percent + quality combined | Values encoded as `'42A'`, `'17B'` | Split: `\d+` → Percent, `[A-E]` → Data Quality |
| Age group & gender not in rows | Stored in pre-header metadata rows 8–9 | Regex extract from raw lines before table parse |

---


## Step 1 — Imports & Config


In [ ]:
import pandas as pd
import sqlite3
import re
from pathlib import Path
from io import StringIO

DATA_DIR   = Path(r"C:\Users\USER\Downloads\Nova-Scotia_Health_Demographic_Trends\Data\Education")
OUTPUT_CSV = Path(r"C:\Users\USER\Downloads\Nova-Scotia_Health_Demographic_Trends\Clean\education_attainment_clean.csv")
SQLITE_DB  = Path(r"C:\Users\USER\Downloads\Nova-Scotia_Health_Demographic_Trends\database\education.db")

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
SQLITE_DB.parent.mkdir(parents=True, exist_ok=True)


## Step 2 — StatsCan Table Reader

Each raw CSV export has a 3-part structure:
1. **Metadata block** (rows 0–7): title, frequency, table number, release date
2. **Filter rows** (rows 8–9): age group and gender — these are not column headers, just context metadata
3. **Data table** (rows 10 onwards): `Geography | Education attainment level 10 | 2012 … 2025`

The reader extracts age group and gender via regex from the metadata block, then locates the data table by scanning for the row that contains both `'Geography'` and `'Education'`. The `,,Percent,,` sub-header row immediately following the column header row is skipped before parsing.


In [ ]:
def read_statscan_table(file_path):
    """
    Reads a raw StatsCan CSV export and returns:
      - A parsed DataFrame of the wide data table
      - The age group string  (e.g. '25 to 34 years')
      - The gender string     (e.g. 'M' or 'F')
    """
    with open(file_path, encoding="utf-8-sig", errors="ignore") as f:
        lines = f.readlines()

    # ── Extract age group and gender from metadata rows 8 & 9 ─────────────
    age_group = "Unknown"
    gender    = "Unknown"

    for line in lines:
        ag_match = re.search(r'"Age group".*?"(\d+ to \d+ years)"', line)
        if ag_match:
            age_group = ag_match.group(1)

        g_match = re.search(r'"Gender[^"]*".*?"(Men|Women)', line, re.IGNORECASE)
        if g_match:
            gender = "M" if g_match.group(1).lower().startswith("men") else "F"

        if age_group != "Unknown" and gender != "Unknown":
            break

    # ── Find the header row and the end of the data block ─────────────────
    start_idx = None
    end_idx   = None

    for i, line in enumerate(lines):
        if start_idx is None and "Geography" in line and "Education" in line:
            start_idx = i
        elif start_idx is not None and line.strip() == "":
            end_idx = i
            break

    if start_idx is None:
        raise ValueError(f"Could not find data table in {file_path}")
    if end_idx is None:
        end_idx = len(lines)

    # ── Skip the ',,Percent,,…' sub-header row ────────────────────────────
    table_lines = [lines[start_idx]] + [
        l for l in lines[start_idx + 1 : end_idx]
        if not re.match(r'^,+"Percent"', l.strip()) and
           not re.match(r'^,+Percent', l.strip())
    ]

    table_text = "".join(table_lines)
    df = pd.read_csv(StringIO(table_text), header=0, dtype=str)

    return df, age_group, gender


## Step 3 — Process One File

For each CSV file:
1. Call `read_statscan_table()` to get the wide DataFrame plus age group and gender
2. Forward-fill `Geography` — StatsCan only writes the province name on the first of its 3 education-level rows
3. Melt wide → long on year columns
4. Split the combined `'42A'` values into `Percent` (integer) and `Data Quality` (letter grade)
5. Strip footnote numbers from Geography names (`'Canada 11'` → `'Canada'`)
6. Tag each row with `Age Group` and `Gender` from the metadata


In [ ]:
def process_file(file_path):

    print(f"   🧹 Processing: {Path(file_path).name}")

    df, age_group, gender = read_statscan_table(file_path)

    # ── Standardise column names ──────────────────────────────────────────
    df.columns = df.columns.str.strip()

    df.rename(columns={
        df.columns[0]: "Geography",
        df.columns[1]: "Education attainment level 10",
    }, inplace=True)

    # ── Year columns = all 4-digit numeric column names ───────────────────
    year_cols = [c for c in df.columns if re.fullmatch(r"\d{4}", str(c).strip())]

    df = df[["Geography", "Education attainment level 10"] + year_cols].copy()

    # ── Forward-fill Geography across the 3 education-level rows ─────────
    df["Geography"] = df["Geography"].replace("", pd.NA).ffill()

    # ── Drop rows with no education level ────────────────────────────────
    df = df[df["Education attainment level 10"].notna()]
    df = df[df["Education attainment level 10"].str.strip() != ""]

    # ── Wide → long ───────────────────────────────────────────────────────
    df = df.melt(
        id_vars=["Geography", "Education attainment level 10"],
        value_vars=year_cols,
        var_name="Year",
        value_name="Raw",
    )

    # ── Split '42A' → Percent=42, Data Quality='A' ────────────────────────
    df["Percent"]      = pd.to_numeric(df["Raw"].str.extract(r"(\d+)")[0], errors="coerce")
    df["Data Quality"] = df["Raw"].str.extract(r"([A-E]$)")[0]

    # ── Strip footnote numbers from Geography names ───────────────────────
    # e.g. 'Canada 11' → 'Canada'
    df["Geography"] = df["Geography"].str.replace(r"\s*\d+$", "", regex=True).str.strip()

    # ── Add metadata columns ──────────────────────────────────────────────
    df["Age Group"] = age_group
    df["Gender"]    = gender
    df["Year"]      = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

    df = df.dropna(subset=["Percent", "Year"])

    print(f"   ✅ {len(df)} rows  |  Age group: {age_group}  |  Gender: {gender}")

    return df[[
        "Geography",
        "Year",
        "Age Group",
        "Gender",
        "Education attainment level 10",
        "Percent",
        "Data Quality",
    ]]


## Step 4 — Run Pipeline & Export


In [ ]:
def run_pipeline():

    print("🚀 Education Attainment ETL Starting\n")

    files = sorted(DATA_DIR.glob("*.csv"))
    print(f"📂 Found {len(files)} CSV files:\n")

    frames = []
    for f in files:
        frames.append(process_file(f))

    final_df = pd.concat(frames, ignore_index=True)

    # Remove exact duplicates that arise when StatsCan files overlap
    final_df = final_df.drop_duplicates()

    final_df = final_df.sort_values(["Geography", "Year", "Age Group", "Gender"])

    # ── Save CSV ───────────────────────────────────────────────────────────
    final_df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Clean CSV saved:  {OUTPUT_CSV}")

    # ── Save SQLite ────────────────────────────────────────────────────────
    conn = sqlite3.connect(SQLITE_DB)
    cursor = conn.cursor()
    cursor.execute("DROP TABLE IF EXISTS education_attainment;")
    conn.commit()
    final_df.to_sql("education_attainment", conn, index=False)
    conn.close()
    print(f"✅ SQLite DB saved:  {SQLITE_DB}")

    print(f"\n📊 Total rows: {len(final_df)}")
    print(f"\nSample:\n{final_df.head(10).to_string(index=False)}")

    return final_df


final_df = run_pipeline()


## ✅ Output Summary

| Output | Location | Rows | Years |
|--------|----------|------|-------|
| Clean CSV | `./Clean/education_attainment_clean.csv` | ~4,651 | 2012–2025 |
| SQLite DB | `./database/education.db` | ~4,651 | 2012–2025 |

**Final schema:** `Geography` \| `Year` \| `Age Group` \| `Gender` \| `Education attainment level 10` \| `Percent` \| `Data Quality`

**Education levels captured:**
- Below upper secondary
- Upper secondary and post-secondary non-tertiary
- Tertiary education

**Feeds into:**
- 📊 Power BI — Tertiary attainment rate: NS vs. Canada trend line (2012–2025)
- 📊 Power BI — Attainment gap by gender across age groups
- 📊 Power BI — Provincial comparison: below upper secondary rates

> ⚠️ **Data Quality Note:** Letter grades (A–E) indicate the coefficient of variation (CV) of each estimate. Grade `D` (CV 25–35%) and `E` (CV > 35%) should be treated with caution in trend analysis — particularly for territories (Yukon, NWT, Nunavut) where small populations produce high sampling variability.

---

*Nova Scotia Health & Population Analytics · Statistics Canada Labour Force Survey — Table 37-10-0130-01*
